In [ ]:
import kagglehub



# Download latest version

path = kagglehub.dataset_download("mohammad2012191/q3-ka-ai-2026")



print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
# Task 1: Write your code here:
import os
import pandas as pd
csv_path=os.path.join(path,"/kaggle/input/q3-ka-ai-2026/Q3_data.csv")
df=pd.read_csv(csv_path)

In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 1: Write your code here: Handle missing values appropriately
numeric_cols = df.select_dtypes(include=['int64', 'float64']).columns
categorical_cols = df.select_dtypes(include=['object']).columns

df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].median())

for col in categorical_cols:
    df[col] = df[col].fillna(df[col].mode()[0])

In [ ]:
# Task 2: Write your code here: Check and remove duplicates if any exist
duplicates = df.duplicated().sum()
print("Number of duplicate rows:", duplicates)

if duplicates > 0:
    df = df.drop_duplicates()
    print("Duplicates is removed")

In [ ]:
# Task 3: Write your code here: Encode categorical variables if needed
#if len(categorical_cols) > 0:
#    df = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

#No Need
df.head()

In [ ]:
# Task 4: Write your code here: Apply feature scaling to numerical features (Use StandardScaler)
target_col = "Target"
from sklearn.preprocessing import StandardScaler
X = df.drop(columns=[target_col])
y = df[target_col]
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X = pd.DataFrame(X_scaled, columns=X.columns)


In [ ]:
# Task 5: Write your code here: Check for target imbalance and state if it is imbalanced or not
counts = df[target_col].value_counts()
percentages = df[target_col].value_counts(normalize=True) * 100

print("Value Counts:\n", counts)
print("\nPercentage Distribution (%):\n", percentages)

if percentages.min() < 20:
    print("IMBALANCED.")
else:
    print("BALANCED")

In [ ]:
!pip install catboost

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, accuracy_score
from catboost import CatBoostClassifier
import numpy as np

In [ ]:
# Task 1: Write your code here:Split the dataset into features (X) and target (y)
target_col = "Target"
X = df.drop(columns=[target_col])
y = df[target_col]

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
from catboost import CatBoostClassifier
import numpy as np

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
f1_scores = []

for train_idx, test_idx in skf.split(X, y):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    model = CatBoostClassifier(
        iterations=200,
        depth=6,
        learning_rate=0.1,
        loss_function='Logloss',
        verbose=0,
        random_state=42,
        early_stopping_rounds=30
    )

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    score = f1_score(y_test, y_pred)
    f1_scores.append(score)

print("Average F1 Score across folds:", np.mean(f1_scores))





In [ ]:
# Feature importance
#Plot feature importance from your trained model
import matplotlib.pyplot as plt
import numpy as np
feature_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(feature_importance['feature'], feature_importance['importance'])
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
#Plot feature importance from your trained model
TOP_N = 10
feature_importance = pd.DataFrame({
    'feature': X.columns,
    'importance': model.get_feature_importance()
}).sort_values('importance', ascending=False)


top_features = feature_importance.head(TOP_N)

plt.figure(figsize=(10, 6))
plt.barh(top_features['feature'], top_features['importance'], color='pink')
plt.xlabel('Importance')
plt.title(f'Top {TOP_N} Most Important Features')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()


In [ ]:
# Task 2: Write your code here: Identify and print the name of the most important feature (the 'golden feature')
golden_feature = feature_names[indices[0]]
print("dthe Golden Feature is:", golden_feature)

In [ ]:
# Task Bonus: Write your code here: